# Clasificador de Imágenes — CIFAR-10
**TP Grupal — Procesamiento de Aprendizaje Automático**

Integrantes: *(completar apellidos/nombres)*  
Carrera: *(completar)*  
IFTS Nro: *(completar)*

---

### Decisiones del grupo

Se optó por **modificar el código original** para obtener mejores predicciones. Las mejoras aplicadas son:

| Parámetro | Original | Modificado | Justificación |
|---|---|---|---|
| Filtros 2da Conv2D | 32 | 64 | Mayor capacidad para extraer features complejas |
| Neuronas capa Dense | 64 | 128 | Representación más rica antes de clasificar |
| Dropout | ❌ | 0.4 | Regularización: reduce overfitting |
| Activación final | logits | softmax | Salida en probabilidades reales (0–1) |
| Epochs | 3 | 20 | 3 epochs es insuficiente para CIFAR-10 |
| Learning rate | default | 0.001 | Control explícito del paso de optimización |
| Batch size | 64 | 64 | Se mantiene |
| Data Augmentation | ❌ | ✅ | Variaciones artificiales mejoran generalización |

El modelo entrenado se exporta en formato **TensorFlow.js** para ser usado desde una interfaz web en JavaScript.

## 1. Instalación de dependencias

In [ ]:
# Instalamos tensorflowjs para poder convertir el modelo al formato que usa el browser
!pip install tensorflowjs --quiet

## 2. Importación de librerías

In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {len(tf.config.list_physical_devices("GPU")) > 0}')

## 3. Carga y preprocesamiento del dataset

CIFAR-10 contiene **60.000 imágenes** de 32×32 píxeles en color (3 canales RGB), distribuidas en 10 clases:
- 50.000 para entrenamiento
- 10.000 para verificación

La **normalización** divide cada valor de píxel (0–255) por 255, dejando todos los valores entre 0 y 1. Esto acelera el entrenamiento y mejora la convergencia del optimizador.

In [ ]:
(imagenes_entrenamiento, etiquetas_entrenamiento), (imagenes_verificacion, etiquetas_verificacion) = \
    keras.datasets.cifar10.load_data()

print('Forma del conjunto de entrenamiento:', imagenes_entrenamiento.shape)
print('Forma del conjunto de verificación: ', imagenes_verificacion.shape)

# Normalización entre 0 y 1
imagenes_entrenamiento = imagenes_entrenamiento / 255.0
imagenes_verificacion  = imagenes_verificacion  / 255.0

# Nombres de las 10 clases
nombres_clases = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                  'dog', 'frog', 'horse', 'ship', 'truck']

## 4. Visualización de ejemplos del dataset

In [ ]:
def mostrar():
    plt.figure(figsize=(8, 4))
    for i in range(16):
        plt.subplot(2, 8, i + 1)
        plt.xticks([])
        plt.yticks([])
        plt.grid(False)
        plt.imshow(imagenes_entrenamiento[i])
        plt.xlabel(nombres_clases[etiquetas_entrenamiento[i][0]], fontsize=7)
    plt.suptitle('Ejemplos del dataset CIFAR-10', fontsize=10)
    plt.tight_layout()
    plt.show()

mostrar()

## 5. Data Augmentation (MEJORA AGREGADA)

El **data augmentation** genera variaciones artificiales de las imágenes de entrenamiento (rotaciones, flips, zoom). Esto expone al modelo a más casos posibles y reduce el overfitting, mejorando la capacidad de generalización.

Se aplica **solo al conjunto de entrenamiento**, nunca al de verificación.

In [ ]:
data_augmentation = keras.Sequential([
    keras.layers.RandomFlip('horizontal'),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
], name='data_augmentation')

# Visualizar efecto del augmentation
plt.figure(figsize=(8, 3))
imagen_ejemplo = imagenes_entrenamiento[0:1]
for i in range(8):
    ax = plt.subplot(1, 8, i + 1)
    img_aug = data_augmentation(imagen_ejemplo, training=True)
    plt.imshow(img_aug[0])
    plt.axis('off')
plt.suptitle('Data augmentation — variaciones de una misma imagen', fontsize=9)
plt.tight_layout()
plt.show()

## 6. Definición del modelo mejorado

**Arquitectura:**
- **Data augmentation** integrado al modelo
- **2 bloques Conv2D → MaxPooling** para extracción jerárquica de features
  - 1er bloque: 32 filtros — detecta bordes, texturas simples
  - 2do bloque: 64 filtros — detecta formas y patrones más complejos
- **Flatten** para aplanar el tensor 3D a 1D
- **Dense(128, relu)** — capa de clasificación intermedia
- **Dropout(0.4)** — desactiva el 40% de neuronas al azar durante el entrenamiento (regularización)
- **Dense(10, softmax)** — salida: probabilidades para cada una de las 10 clases

In [ ]:
capa = keras.layers

modelo = keras.models.Sequential([
    # Augmentation (solo activo durante entrenamiento)
    data_augmentation,

    # Bloque convolucional 1 — igual que el original
    capa.Conv2D(32, (3, 3), strides=(1, 1), padding='valid',
                activation='relu', input_shape=(32, 32, 3)),
    capa.MaxPool2D(2, 2),

    # Bloque convolucional 2 — 64 filtros en lugar de 32 (MEJORA)
    capa.Conv2D(64, 3, activation='relu'),
    capa.MaxPool2D(2, 2),

    capa.Flatten(),

    # Capa densa 128 neuronas en lugar de 64 (MEJORA)
    capa.Dense(128, activation='relu'),

    # Dropout para regularización (MEJORA)
    capa.Dropout(0.4),

    # Salida con softmax — produce probabilidades reales (MEJORA vs logits)
    capa.Dense(10, activation='softmax')
])

modelo.summary()

## 7. Compilación del modelo

- **Optimizador Adam** con learning rate 0.001 explícito: algoritmo adaptativo, converge más rápido que SGD
- **Loss: SparseCategoricalCrossentropy** — función de pérdida estándar para clasificación multiclase con etiquetas enteras
- **Métrica: accuracy** — proporción de predicciones correctas

In [ ]:
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=['accuracy']
)

## 8. Entrenamiento

Se entrena con **20 epochs** (el original usaba 3 — insuficiente para CIFAR-10).

Se usa `EarlyStopping` como callback: si el `val_accuracy` no mejora en 5 epochs seguidas, el entrenamiento se detiene automáticamente y se restauran los mejores pesos. Evita overfitting y desperdicio de tiempo.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

historial = modelo.fit(
    imagenes_entrenamiento,
    etiquetas_entrenamiento,
    epochs=20,
    batch_size=64,
    validation_data=(imagenes_verificacion, etiquetas_verificacion),
    callbacks=[early_stopping],
    verbose=1
)

## 9. Evaluación del modelo

In [ ]:
loss, accuracy = modelo.evaluate(imagenes_verificacion, etiquetas_verificacion,
                                  batch_size=64, verbose=0)
print(f'Loss final:    {loss:.4f}')
print(f'Accuracy final: {accuracy:.4f} ({accuracy*100:.1f}%)')

## 10. Curvas de entrenamiento

Visualizamos cómo evolucionó el **accuracy** y el **loss** a lo largo de los epochs, tanto para entrenamiento como para validación. Una brecha grande entre ambas curvas indica overfitting.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy
ax1.plot(historial.history['accuracy'],     label='Entrenamiento', linewidth=2)
ax1.plot(historial.history['val_accuracy'], label='Validación',     linewidth=2, linestyle='--')
ax1.set_title('Accuracy por epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(historial.history['loss'],     label='Entrenamiento', linewidth=2, color='orange')
ax2.plot(historial.history['val_loss'], label='Validación',     linewidth=2, linestyle='--', color='red')
ax2.set_title('Loss por epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Predicción — CONTINUAR DESDE ACA

La función `predecir()` recibe una imagen (array NumPy), la preprocesa y devuelve la clase predicha con su probabilidad.

In [ ]:
def predecir(imagen):
    """
    Predice la clase de una imagen.
    - imagen: array NumPy de shape (H, W, 3), valores en [0, 1] o [0, 255]
    Retorna: (clase_predicha, probabilidad, array_de_probabilidades)
    """
    # Redimensionar a 32x32 si es necesario
    img = tf.image.resize(imagen, [32, 32]).numpy()

    # Normalizar si los valores están en [0, 255]
    if img.max() > 1.0:
        img = img / 255.0

    # Agregar dimensión de batch: (1, 32, 32, 3)
    img_batch = np.expand_dims(img, axis=0)

    # Predicción
    probabilidades = modelo.predict(img_batch, verbose=0)[0]
    indice = np.argmax(probabilidades)

    return nombres_clases[indice], probabilidades[indice], probabilidades


# --- Prueba con imágenes del conjunto de verificación ---
n = 12
indices = np.random.choice(len(imagenes_verificacion), n, replace=False)

plt.figure(figsize=(14, 5))
for i, idx in enumerate(indices):
    clase_real      = nombres_clases[etiquetas_verificacion[idx][0]]
    clase_pred, prob, _ = predecir(imagenes_verificacion[idx])
    correcto = clase_real == clase_pred

    ax = plt.subplot(2, 6, i + 1)
    plt.imshow(imagenes_verificacion[idx])
    color = 'green' if correcto else 'red'
    plt.title(f'Pred: {clase_pred}\n({prob*100:.0f}%)', fontsize=7, color=color)
    plt.xlabel(f'Real: {clase_real}', fontsize=7)
    plt.xticks([])
    plt.yticks([])

plt.suptitle('Predicciones — verde: correcto, rojo: incorrecto', fontsize=9)
plt.tight_layout()
plt.show()

## 12. Subir una imagen propia para predecir

**Botón para cargar una imagen local** — el usuario puede subir cualquier imagen y el modelo la clasifica.

In [ ]:
from google.colab import files
from PIL import Image
import io

print('Seleccioná una imagen de tu computadora:')
archivos = files.upload()

for nombre_archivo, contenido in archivos.items():
    imagen_pil = Image.open(io.BytesIO(contenido)).convert('RGB')
    imagen_np  = np.array(imagen_pil) / 255.0

    clase, prob, todas_probs = predecir(imagen_np)

    # Mostrar imagen y resultado
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.imshow(imagen_pil)
    ax1.set_title(f'Imagen: {nombre_archivo}', fontsize=9)
    ax1.axis('off')

    # Barras de probabilidad
    colores = ['#1D9E75' if c == clase else '#B4B2A9' for c in nombres_clases]
    barras  = ax2.barh(nombres_clases, todas_probs * 100, color=colores)
    ax2.set_xlabel('Probabilidad (%)')
    ax2.set_title(f'Predicción: {clase} ({prob*100:.1f}%)', fontsize=10, fontweight='bold')
    ax2.set_xlim(0, 100)

    for barra, p in zip(barras, todas_probs):
        ax2.text(barra.get_width() + 0.5, barra.get_y() + barra.get_height()/2,
                 f'{p*100:.1f}%', va='center', fontsize=8)

    plt.tight_layout()
    plt.show()

    print(f'\nArchivo: {nombre_archivo}')
    print(f'Predicción: {clase.upper()} ({prob*100:.1f}% de confianza)')

## 13. Exportación para TensorFlow.js

Guardamos el modelo en formato `.h5` y luego lo convertimos al formato que consume el browser (archivos `model.json` + pesos en `.bin`).

In [ ]:
import os

# 1. Guardar modelo en formato Keras
modelo.save('cifar10_modelo.h5')
print('Modelo guardado como cifar10_modelo.h5')

# 2. Convertir a formato TensorFlow.js
os.makedirs('tfjs_model', exist_ok=True)
!tensorflowjs_converter \
    --input_format=keras \
    cifar10_modelo.h5 \
    tfjs_model/

print('\nArchivos generados en tfjs_model/:')
for f in os.listdir('tfjs_model/'):
    size_kb = os.path.getsize(f'tfjs_model/{f}') / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

In [ ]:
# 3. Comprimir y descargar los archivos del modelo TFJS
import shutil
shutil.make_archive('cifar10_tfjs', 'zip', 'tfjs_model')
files.download('cifar10_tfjs.zip')
print('Descargando cifar10_tfjs.zip...')
print('Contenido: model.json + archivos de pesos .bin')
print('Estos archivos van en la misma carpeta que tu HTML/JS.')

## 14. Instrucciones para usar el modelo en JavaScript

Una vez descargado el `.zip`, descomprimilo en la misma carpeta que el archivo HTML.

Luego en tu JS, cargás el modelo así:

```javascript
import * as tf from '@tensorflow/tfjs';

// Cargar el modelo exportado desde Colab
const model = await tf.loadLayersModel('./tfjs_model/model.json');

// Preprocesar y predecir
const tensor = tf.browser.fromPixels(imgElement)
    .resizeBilinear([32, 32])
    .toFloat()
    .div(255.0)
    .expandDims(0);

const probs = model.predict(tensor);
const topClass = probs.argMax(1).dataSync()[0];
```